In [ ]:
!wget https://storage.googleapis.com/zerospeech-checkpoints/samples/1272-128104-0000.flac
!pip install zerosyl

In [ ]:
import torch
import torchaudio
from IPython.display import Audio
from zerosyl import ZeroSylCollapsed

In [ ]:
# ------------------------ Colab Hack -----------------------------
# Some of the torch hub models' weights are stored as CUDA tensor.
# This hack ensures they are loaded to CPU and work in Google Colab.
# You can delete this cell if you are running on a CUDA machine.
if not torch.cuda.is_available() and "_original_hub_load" not in globals():
    _original_hub_load = torch.hub.load_state_dict_from_url

    def _safe_hub_load(*args, **kwargs):
        new_args = list(args)
        if len(new_args) > 2:
            new_args[2] = "cpu"
        else:
            kwargs["map_location"] = "cpu"
        return _original_hub_load(*new_args, **kwargs)

    torch.hub.load_state_dict_from_url = _safe_hub_load
# -------------------------------------------------------------------

In [ ]:
num_decoding_steps = 10
temperature = 1.0
top_p = 0.85
device = "cuda" if torch.cuda.is_available() else "cpu"
waveform_path = "1272-128104-0000.flac"

In [ ]:
wav, sr = torchaudio.load(waveform_path)

zerosyl = ZeroSylCollapsed.from_remote()

maskgit = torch.hub.load(
    "nicolvisser/S2SMaskGIT:master", "s2smaskgit", trust_repo=True
).to(device)

acoustic = torch.hub.load(
    "bshall/acoustic-model:main", "hubert_discrete", trust_repo=True
).to(device)

hifigan = torch.hub.load(
    "bshall/hifigan:main", "hifigan_hubert_discrete", trust_repo=True
).to(device)

In [ ]:
with torch.inference_mode():

    seg_starts, seg_ends, seg_ids = zerosyl.encode(wav)
    seg_lengths = seg_ends - seg_starts
    semantic_units = torch.repeat_interleave(seg_ids, seg_lengths)

    acoustic_units = maskgit.generate(
        semantic_units, num_decoding_steps, temperature, top_p
    )

    mel = acoustic.generate(acoustic_units.unsqueeze(0)).transpose(1, 2)

    target = hifigan(mel).squeeze(0).cpu()

In [ ]:
acoustic_units

In [ ]:
Audio(target, rate=16000)